In [2]:
# pip install -r requirements.txt

In [3]:
import os
import warnings
import numpy as np
import pandas as pd

# 1. Deactivate system warnings for clean execution logs
warnings.filterwarnings('ignore')

# 2. Establish Global Stochastic Control (For reproducibility in data processing)
os.environ['PYTHONHASHSEED'] = '0'
np.random.seed(42)


In [4]:
df_coffee = pd.read_csv('Data/Daily/coffee_price.csv')
df_coffee.head()

,Ngày,Lần cuối,Mở,Cao,Thấp,KL,% Thay đổi
0,28/01/2026,"4,148.00","4,330.00","4,330.00","4,098.00",NaN,1.17%
1,27/01/2026,"4,100.00","4,022.00","4,127.00","3,993.00",4.87K,-0.32%
2,26/01/2026,"4,113.00","4,039.00","4,145.00","4,026.00",9.36K,1.53%
3,23/01/2026,"4,051.00","3,947.00","4,114.00","3,939.00",13.26K,2.74%
4,22/01/2026,"3,943.00","3,960.00","3,974.00","3,921.00",9.38K,-1.03%


In [5]:
# Transfer "Ngay" column to datetime format
df_coffee['Ngày'] = pd.to_datetime(df_coffee['Ngày'], format='%d/%m/%Y')
df_coffee.head()

,Ngày,Lần cuối,Mở,Cao,Thấp,KL,% Thay đổi
0,2026-01-28,"4,148.00","4,330.00","4,330.00","4,098.00",NaN,1.17%
1,2026-01-27,"4,100.00","4,022.00","4,127.00","3,993.00",4.87K,-0.32%
2,2026-01-26,"4,113.00","4,039.00","4,145.00","4,026.00",9.36K,1.53%
3,2026-01-23,"4,051.00","3,947.00","4,114.00","3,939.00",13.26K,2.74%
4,2026-01-22,"3,943.00","3,960.00","3,974.00","3,921.00",9.38K,-1.03%


In [6]:
# Data parsing: Remove thousand separators and cast target variable to float.
df_coffee['Lần cuối'] = df_coffee['Lần cuối'].str.replace(',', '').astype(float)
df_coffee.head()

,Ngày,Lần cuối,Mở,Cao,Thấp,KL,% Thay đổi
0,2026-01-28,4148.0,"4,330.00","4,330.00","4,098.00",NaN,1.17%
1,2026-01-27,4100.0,"4,022.00","4,127.00","3,993.00",4.87K,-0.32%
2,2026-01-26,4113.0,"4,039.00","4,145.00","4,026.00",9.36K,1.53%
3,2026-01-23,4051.0,"3,947.00","4,114.00","3,939.00",13.26K,2.74%
4,2026-01-22,3943.0,"3,960.00","3,974.00","3,921.00",9.38K,-1.03%


In [7]:
# Feature selection: Isolate Target variable (Close Price) and Volume.
df_coffee = df_coffee[['Ngày', 'Lần cuối', 'KL']]
df_coffee.head()

,Ngày,Lần cuối,KL
0,2026-01-28,4148.0,NaN
1,2026-01-27,4100.0,4.87K
2,2026-01-26,4113.0,9.36K
3,2026-01-23,4051.0,13.26K
4,2026-01-22,3943.0,9.38K


In [8]:
df_coffee = df_coffee.rename(columns={'Ngày': 'Date', 'Lần cuối': 'Last Price', 'KL': 'Volume'})

In [9]:
df_coffee = df_coffee.sort_values('Date').reset_index(drop=True)
df_coffee.head()

,Date,Last Price,Volume
0,2008-01-14,2051.0,NaN
1,2008-01-15,2036.0,NaN
2,2008-01-16,2040.0,0.00K
3,2008-01-17,2037.0,NaN
4,2008-01-18,2027.0,NaN


In [10]:
def clean_volume(vol): 
    if pd.isna(vol) or vol == '':
        return 0.0
    vol = str(vol).strip().upper()
    mult = 1.0 
    if 'K' in vol:
        mult = 1000.0
        vol = vol.replace('K', '')
    elif 'M' in vol:
        mult = 1000000.0
        vol = vol.replace('M', '')

    try:
        return float(vol) * mult
    except ValueError:
        return 0.0

In [11]:
df_coffee['Volume'] = df_coffee['Volume'].apply(clean_volume)
df_coffee.tail()

,Date,Last Price,Volume
4598,2026-01-22,3943.0,9380.0
4599,2026-01-23,4051.0,13260.0
4600,2026-01-26,4113.0,9360.0
4601,2026-01-27,4100.0,4870.0
4602,2026-01-28,4148.0,0.0


Oke done coffee data

In [12]:
def load_macro_data(filepath, value_col_name):
    
    df = pd.read_csv(filepath)
    # Đồng bộ hóa tên cột ngày
    df['Date'] = pd.to_datetime(df['observation_date'])
    
    val_col = df.columns[1] # Lấy cột giá trị
    
    # Xử lý dấu phẩy thập phân 
    if df[val_col].dtype == 'string':
        df[val_col] = df[val_col].astype(str).str.replace(',', '.').astype(float)
        
    df = df[['Date', val_col]]
    df.rename(columns={val_col: value_col_name}, inplace=True)
    return df

# load 3 file
df_usd = load_macro_data('Data/Daily/USD_INDEX.csv', 'USD_Index')
df_effr = load_macro_data('Data/Daily/EFFR.csv', 'EFFR')
df_oil = load_macro_data('Data/Daily/Crude Oil Prices Daily.csv', 'Crude_Oil')

display(df_usd.head())
display(df_effr.head())
display(df_oil.head())


,Date,USD_Index
0,2008-01-14,89.2749
1,2008-01-15,89.1555
2,2008-01-16,89.6701
3,2008-01-17,89.6484
4,2008-01-18,89.8993


,Date,EFFR
0,2008-01-14,4.24
1,2008-01-15,4.24
2,2008-01-16,4.22
3,2008-01-17,4.23
4,2008-01-18,4.17


,Date,Crude_Oil
0,2008-01-14,94.23
1,2008-01-15,91.87
2,2008-01-16,90.80
3,2008-01-17,90.11
4,2008-01-18,90.55


In [21]:
# 1. Cross-market data merging based on trading calendar of the primary market (ICE London)
df_final = df_coffee.merge(df_usd, on='Date', how='left')
df_final = df_final.merge(df_effr, on='Date', how='left')
df_final = df_final.merge(df_oil, on='Date', how='left')

# 2. Handling asynchronous market calendars via Forward-fill (ffill) imputation
# This accounts for localized bank holidays where macro indicators remain unchanged from the prior trading session.
df_final = df_final.ffill()

# 3. Drop remaining initial NaN rows resulting from the absence of antecedent lookback data
df_final = df_final.dropna()

# 4. Export synchronized multivariate dataset for hybrid modeling
# df_final.to_csv('Cleaned_Multivariate_Data.csv', index=False)

df_final

,Date,Last Price,Volume,USD_Index,EFFR,Crude_Oil
0,2008-01-14,2051.0,0.0,89.2749,4.24,94.23
1,2008-01-15,2036.0,0.0,89.1555,4.24,91.87
2,2008-01-16,2040.0,0.0,89.6701,4.22,90.80
3,2008-01-17,2037.0,0.0,89.6484,4.23,90.11
4,2008-01-18,2027.0,0.0,89.8993,4.17,90.55
...,...,...,...,...,...,...
4598,2026-01-22,3943.0,9380.0,119.1962,3.64,59.24
4599,2026-01-23,4051.0,13260.0,118.8976,3.64,60.70
4600,2026-01-26,4113.0,9360.0,118.0525,3.64,60.46
4601,2026-01-27,4100.0,4870.0,117.4523,3.64,62.04
